# Update T-Route channel geometry: Arkansas-White-Red region (HUC2 11), a = 0.4

T-Route's Muskingum-Cunge routing treats every reach as a trapezoidal channel, defined in the NWM RouteLink file. This notebook replaces that geometry with ML-predicted bankfull geometry from [HydroGeom](https://github.com/Reizrb/HydroGeom) for the reaches in **HUC2 11**, using a trapezoid whose bottom width is **0.4 × top width**, and recomputes the Muskingum parameters.

You need the original NWM v3.0 RouteLink file (`RouteLink_CONUS.nc`). It is never modified; a new file is written.

### STEP 1. Install and import

Install once:
```bash
pip install numpy pandas netCDF4 "git+https://github.com/Reizrb/HydroGeom"
```

In [ ]:
import sys
sys.path.append("..")          # troute_geometry.py is in the repository's main folder

import pandas as pd
import troute_geometry as tg

### STEP 2. Settings

- `routelink_in`: path to the original RouteLink file
- `a`: bottom width / top width of the trapezoid (0 = triangle; use 0.99 for a nearly rectangular channel)
- `huc2`: region whose reaches get the new geometry (all other reaches keep the original geometry)

In [ ]:
routelink_in = "RouteLink_CONUS.nc"
a = 0.4
huc2 = "11"

routelink_out = f"RouteLink_CONUS_HUC2-{huc2}_a{a}.nc"

### STEP 3. Bankfull geometry from HydroGeom

Bankfull width and depth for every NHDPlusV2.1 reach in the region. For all of CONUS, download the dataset from [Zenodo](https://doi.org/10.5281/zenodo.19208847) and use `tg.read_geometry_file(path)` instead.

In [ ]:
geometry = tg.channel_geometry(huc2=huc2)
print(f"{len(geometry):,} reaches with bankfull geometry in HUC2 {huc2}")

### STEP 4. Write the updated RouteLink file

Sets `BtmWdth`, `TopWdth`, and `ChSlp` for the trapezoid, and recomputes `TopWdthCC`, `MusK`, and `MusX`.

In [ ]:
tg.update_routelink(routelink_in, routelink_out, geometry, a=a)

### STEP 5. Original vs updated values for the reaches in the region

In [ ]:
pd.set_option("display.width", 160)
tg.compare(routelink_in, routelink_out, comids=geometry["COMID"])

### STEP 6. Use it in T-Route

Point T-Route to the new file in its configuration YAML:

```yaml
network_topology_parameters:
  supernetwork_parameters:
    geo_file_path: RouteLink_CONUS_HUC2-11_a0.4.nc
```